# Parte 2 — Pipeline Reutilizable
### Workshop: Clasificación de Emociones en Twitter

Este notebook define las funciones compartidas que usan todos los experimentos:
- Métricas (`compute_metrics`)
- Tokenización (`make_tokenized_dataset`)
- Evaluación completa (`full_evaluation`)
- Curvas de entrenamiento (`plot_training_curves`)
- Constructor de Trainer (`make_trainer`)

**Prerequisito:** haber ejecutado `part-1-data.ipynb`

> En lugar de repetir código en cada notebook, definimos todo aquí. En la práctica esto se convertiría en un módulo Python importable.

In [1]:
# !pip install 'transformers[torch]' 'accelerate>=1.1.0' peft datasets evaluate scikit-learn matplotlib -q

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import evaluate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 5
LABEL_NAMES = ["anger", "joy", "optimism", "sadness"]
ID2LABEL = {i: l for i, l in enumerate(LABEL_NAMES)}
LABEL2ID = {l: i for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS = len(LABEL_NAMES)

np.random.seed(SEED)
torch.manual_seed(SEED)

# Cargar el dataset una vez — todos los experimentos lo usan
raw = load_dataset("tweet_eval", "emotion")
print(raw)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 3257
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1421
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 374
    })
})


## Métricas

In [6]:
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")
    acc = acc_metric.compute(predictions=preds, references=labels)
    return {"f1_macro": f1["f1"], "accuracy": acc["accuracy"]}

print("compute_metrics OK")

compute_metrics OK


## Tokenización

In [7]:
def make_tokenized_dataset(tokenizer, max_length=MAX_LENGTH):
    """Tokeniza los tres splits con el tokenizador dado."""
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    return raw.map(tokenize, batched=True, remove_columns=["text"])

print("make_tokenized_dataset OK")

make_tokenized_dataset OK


## Evaluación completa

In [8]:
def full_evaluation(trainer, test_dataset, model_name=""):
    """Classification report + confusion matrix para el split de test."""
    output = trainer.predict(test_dataset)
    preds = np.argmax(output.predictions, axis=1)
    labels = output.label_ids

    title = f"Test — {model_name}" if model_name else "Test"
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(classification_report(labels, preds,
                                 target_names=LABEL_NAMES, digits=4))

    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(
        ax=axes[0], colorbar=False, cmap="Blues", xticks_rotation=30)
    axes[0].set_title("Confusion Matrix (conteos)", fontweight="bold")
    ConfusionMatrixDisplay(cm_norm, display_labels=LABEL_NAMES).plot(
        ax=axes[1], colorbar=False, cmap="Blues",
        values_format=".2f", xticks_rotation=30)
    axes[1].set_title("Confusion Matrix (normalizada)", fontweight="bold")
    plt.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

    return output.metrics

print("full_evaluation OK")

full_evaluation OK


## Curvas de entrenamiento

In [9]:
def plot_training_curves(trainer, title=""):
    logs = pd.DataFrame(trainer.state.log_history)
    train_logs = logs[logs["loss"].notna()][["step", "loss", "learning_rate"]]
    eval_logs = logs[logs["eval_loss"].notna()][
        ["epoch", "eval_loss", "eval_f1_macro", "eval_accuracy"]]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(train_logs["step"], train_logs["loss"],
                 color="steelblue", linewidth=1.5)
    axes[0].set_title("Training Loss", fontweight="bold")
    axes[0].set_xlabel("Paso"); axes[0].set_ylabel("Loss")
    axes[0].spines[["top", "right"]].set_visible(False)

    axes[1].plot(eval_logs["epoch"], eval_logs["eval_f1_macro"],
                 marker="o", color="#e74c3c", linewidth=2, label="F1 macro")
    axes[1].plot(eval_logs["epoch"], eval_logs["eval_accuracy"],
                 marker="s", color="#2ecc71", linewidth=2, label="Accuracy")
    axes[1].set_title("Métricas de Validación", fontweight="bold")
    axes[1].set_xlabel("Época"); axes[1].set_ylabel("Score")
    axes[1].set_ylim(0.3, 1.0); axes[1].legend()
    axes[1].spines[["top", "right"]].set_visible(False)

    axes[2].plot(train_logs["step"], train_logs["learning_rate"],
                 color="purple", linewidth=1.5)
    axes[2].set_title("Learning Rate Schedule", fontweight="bold")
    axes[2].set_xlabel("Paso"); axes[2].set_ylabel("LR")
    axes[2].spines[["top", "right"]].set_visible(False)

    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

print("plot_training_curves OK")

plot_training_curves OK


## Constructor de Trainer

In [10]:
def make_trainer(model, tokenizer, tokenized_ds, output_dir,
                 lr=2e-5, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Construye un Trainer con configuración estándar."""
    n_train_steps = (len(tokenized_ds["train"]) // batch_size) * epochs

    args = TrainingArguments(
        output_dir = output_dir,
        num_train_epochs = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size = batch_size,
        learning_rate = lr,
        weight_decay = 0.01,
        warmup_steps = int(0.1 * n_train_steps),
        lr_scheduler_type = "linear",
        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model = "f1_macro",
        greater_is_better = True,
        logging_steps = 20,
        report_to = "none",
        seed = SEED,
        fp16 = torch.cuda.is_available(),
    )

    return Trainer(
        model = model,
        args = args,
        train_dataset = tokenized_ds["train"],
        eval_dataset = tokenized_ds["validation"],
        processing_class = tokenizer,
        data_collator = DataCollatorWithPadding(tokenizer),
        compute_metrics = compute_metrics,
        callbacks = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

print("make_trainer OK")
print("\nPipeline lista. Puedes continuar con part-3-distilbert.ipynb")

make_trainer OK

Pipeline lista. Puedes continuar con part-3-distilbert.ipynb
